# Live Stats Monitor

Panel 3 displays live SOR stats from `/stats/current`, active policy identity from `/policy/current`, and lightweight charts for notebook demo monitoring.


In [1]:
from IPython.display import HTML, display
import pandas as pd
from adaptive_quantum_sor import SorNotebookClient

client = SorNotebookClient()
stats = client.stats_dataframe()
policy = client.policy_dataframe()
snapshot = pd.concat([stats.add_prefix('stats.'), policy.add_prefix('policy.')], axis=1)
snapshot


,stats.policyVersion,stats.venueCount,policy.policyVersion,policy.policyHash64
0,1,100,1,-2649222367500659138


In [2]:
def _first_value(frame, *names, default=0):
    for name in names:
        if name in frame.columns:
            return frame.iloc[0][name]
    return default

kpis = pd.DataFrame([
    {'metric': 'Venue Count', 'value': _first_value(stats, 'venueCount')},
    {'metric': 'Policy Version', 'value': _first_value(policy, 'policyVersion')},
    {'metric': 'Parent Orders', 'value': _first_value(stats, 'parentOrderCount', 'ordersAccepted')},
    {'metric': 'Child Orders', 'value': _first_value(stats, 'childOrderCount', 'ordersRouted')},
])

cards = ''.join(
    f"<div class='sor-kpi'><span>{row.metric}</span><strong>{row.value}</strong></div>"
    for row in kpis.itertuples(index=False)
)
display(HTML(f"""
<style>
.sor-kpi-grid {{display:grid;grid-template-columns:repeat(4,minmax(120px,1fr));gap:12px;margin:8px 0 16px 0;}}
.sor-kpi {{border:1px solid #d6dae1;border-radius:6px;padding:12px 14px;background:#f8fafc;}}
.sor-kpi span {{display:block;color:#4b5563;font-size:12px;text-transform:uppercase;letter-spacing:.04em;}}
.sor-kpi strong {{display:block;color:#111827;font-size:24px;line-height:1.2;margin-top:4px;}}
</style>
<div class='sor-kpi-grid'>{cards}</div>
"""))
kpis


,metric,value
0,Venue Count,100
1,Policy Version,1
2,Parent Orders,0
3,Child Orders,0


In [3]:
chart_rows = []
for frame_name, frame in [('stats', stats), ('policy', policy)]:
    for column in frame.columns:
        value = frame.iloc[0][column]
        if pd.api.types.is_number(value) and not isinstance(value, bool) and pd.notna(value):
            chart_rows.append({'source': frame_name, 'metric': column, 'value': float(value)})

chart = pd.DataFrame(chart_rows, columns=['source', 'metric', 'value']).sort_values(['source', 'metric']).reset_index(drop=True)
if chart.empty:
    display(HTML('<p>No numeric stats are available for charting yet.</p>'))
else:
    scale = max(chart['value'].abs().max(), 1.0)
    chart['bar_width_pct'] = (chart['value'].abs() / scale * 100).round(1)
    bars = ''.join(
        f"<div class='sor-bar-row'><span class='sor-bar-label'>{row.source}.{row.metric}</span>"
        f"<div class='sor-bar-track'><div class='sor-bar' style='width:{row.bar_width_pct}%'></div></div>"
        f"<span class='sor-bar-value'>{row.value:g}</span></div>"
        for row in chart.itertuples(index=False)
    )
    display(HTML(f"""
<style>
.sor-bar-panel {{border:1px solid #d6dae1;border-radius:6px;padding:12px;background:white;}}
.sor-bar-row {{display:grid;grid-template-columns:190px 1fr 90px;gap:10px;align-items:center;margin:7px 0;}}
.sor-bar-label {{font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:12px;color:#374151;}}
.sor-bar-track {{height:12px;background:#eef2f7;border-radius:3px;overflow:hidden;}}
.sor-bar {{height:12px;background:#2563eb;}}
.sor-bar-value {{text-align:right;font-variant-numeric:tabular-nums;color:#111827;}}
</style>
<div class='sor-bar-panel'>{bars}</div>
"""))
chart[['source', 'metric', 'value']]


,source,metric,value
0,policy,policyHash64,-2.649222e+18
1,policy,policyVersion,1.000000e+00
2,stats,policyVersion,1.000000e+00
3,stats,venueCount,1.000000e+02
